# MorphoNet-BT — 01: Environment and Dataset Inspection
## Objective and method
Inspect the dl environment and all actual Figshare files without training, splitting or modifying data. Read archives directly, verify ZIP CRC and document structure before calculating statistics. Expected counts of 3,064 images and 233 patients are comparison targets, not assumed findings. This phase establishes neither model novelty nor normalization choice.

## Public-copy execution policy
This is an English rendering of a historical research notebook. Outputs and execution counts were cleared; no cell was rerun for Phase 18. Historical findings below describe earlier runs only. Every code cell is gated, including setup cells that may perform model checks, preprocessing or writes. Enable `CONFIRM_PUBLIC_WORKFLOW` deliberately in the first code cell only after reviewing [the reproduction guide](../docs/reproducibility.md). Existing stage-specific confirmations must also be enabled separately. Run from the repository root or set `MORPHONET_ROOT` to a private writable experiment workspace with the same source/config layout; set `MORPHONET_DATA_ROOT` for dataset discovery. Original source/configuration fingerprints, private split manifests and checkpoints are not portable artifacts; do not bypass their validation. See the guide for prerequisites and limitations.


## Configuration

In [ ]:
CONFIRM_PUBLIC_WORKFLOW = False  # Explicit opt-in; no cell has been rerun.
if not globals().get("CONFIRM_PUBLIC_WORKFLOW", False):
    raise RuntimeError("Execution disabled. Read docs/reproducibility.md and explicitly enable this workflow.")
from pathlib import Path
PROJECT_ROOT = Path(__import__("os").environ.get("MORPHONET_ROOT", str(Path.cwd()))).resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
assert (PROJECT_ROOT / "src/data/inspection.py").is_file()
DATASET_ROOT = Path(__import__("os").environ.get("MORPHONET_DATA_ROOT", str(PROJECT_ROOT.parent / "1512427")))
RESULTS = PROJECT_ROOT / "results"
RESULTS.mkdir(exist_ok=True)
SEED = 42
assert DATASET_ROOT.is_dir(), f"Dataset missing: {DATASET_ROOT}"
print("Dataset:", DATASET_ROOT)

## Imports and environment verification

In [ ]:
if not globals().get("CONFIRM_PUBLIC_WORKFLOW", False):
    raise RuntimeError("Execution disabled. Read docs/reproducibility.md and explicitly enable this workflow.")
import sys, json, random
sys.path.insert(0, str(PROJECT_ROOT))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from IPython.display import display, Markdown
from src.data.inspection import environment, discover, read_record, inspect_dataset, summarize, CLASSES
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
env = environment()
env["seed"] = SEED
(RESULTS / "environment.json").write_text(json.dumps(env, indent=2), encoding="utf-8")
display(pd.Series(env))

## Dataset discovery and actual structure
Separate files without cjdata from MRI samples; cvind.mat is not assumed patient-safe. The local dataset README maps 1=Meningioma, 2=Glioma, 3=Pituitary. HDF5 reading reverses MATLAB dimension order jointly for image and mask, without resizing or normalization.

In [ ]:
if not globals().get("CONFIRM_PUBLIC_WORKFLOW", False):
    raise RuntimeError("Execution disabled. Read docs/reproducibility.md and explicitly enable this workflow.")
records = discover(DATASET_ROOT)
assert records, "No MATLAB files found"
display(pd.DataFrame(records).groupby("source").size().rename("mat_files"))
sample_record = next(r for r in records if r["member"])
sample, schema = read_record(sample_record)
print(json.dumps(schema, indent=2))
print("PID stored:", sample["PID"].tolist())
print("Label stored:", sample["label"].tolist())
print("Border first coordinates:", sample["tumorBorder"].ravel()[:12])

## Inspect every file and save sample records
Unreadable files are logged, never silently deleted or excluded.

In [ ]:
if not globals().get("CONFIRM_PUBLIC_WORKFLOW", False):
    raise RuntimeError("Execution disabled. Read docs/reproducibility.md and explicitly enable this workflow.")
frame, auxiliary, errors, schemas = inspect_dataset(records)
assert len(frame), "No readable cjdata samples"
summary = summarize(frame, records, auxiliary, errors)
frame.to_csv(RESULTS / "dataset_statistics.csv", index=False)
pd.DataFrame(errors, columns=["source", "member", "sample_id", "error"]).to_csv(RESULTS / "dataset_errors.csv", index=False)
(RESULTS / "dataset_structure.json").write_text(json.dumps(schemas, indent=2), encoding="utf-8")
(RESULTS / "auxiliary_files.json").write_text(json.dumps(auxiliary, indent=2), encoding="utf-8")
(RESULTS / "dataset_summary.txt").write_text(json.dumps(summary, indent=2, ensure_ascii=False), encoding="utf-8")
display(pd.Series(summary))

## Classes, patients and slice counts

In [ ]:
if not globals().get("CONFIRM_PUBLIC_WORKFLOW", False):
    raise RuntimeError("Execution disabled. Read docs/reproducibility.md and explicitly enable this workflow.")
class_table = frame.groupby("class_name").agg(images=("sample_id", "size"), patients=("PID", "nunique"))
patient_table = frame.groupby("PID").agg(slices=("sample_id", "size"), distinct_labels=("raw_label", "nunique"))
class_table.to_csv(RESULTS / "class_distribution.csv")
patient_table.to_csv(RESULTS / "patient_statistics.csv")
display(class_table)
display(patient_table.slices.describe())
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
class_table.images.plot.bar(ax=axes[0], title="MRI slices per class", rot=0)
patient_table.slices.plot.hist(ax=axes[1], bins=20, title="Slices per patient")
fig.tight_layout()
fig.savefig(RESULTS / "dataset_distribution.png", dpi=200)
plt.show()

## Intensity, masks and borders
Intensity statistics are raw; normalization is addressed in Phase 3. Display uses each image’s grayscale range. Check dimensions, binary masks and finite paired border coordinates within the image. Matching shapes do not prove anatomical alignment; examples support visual review.

In [ ]:
if not globals().get("CONFIRM_PUBLIC_WORKFLOW", False):
    raise RuntimeError("Execution disabled. Read docs/reproducibility.md and explicitly enable this workflow.")
display(frame[["image_min", "image_max", "image_mean", "image_std", "image_p01", "image_p99", "mask_pixels", "border_values"]].describe())
display(frame.groupby(["image_shape", "image_dtype", "mask_shape", "mask_dtype"]).size())
print("Unreadable:", errors)
print("Auxiliary:", auxiliary)
print("Inconsistent PID labels:", summary["inconsistent_pids"])
print("Exact image duplicates across patients:", summary["cross_patient_duplicate_image_groups"])

## Representative MRI, tumorMask and tumorBorder examples
Choose the first two samples per class in a fixed order, without quality-based selection. Cyan lines show original tumorBorder coordinates converted from MATLAB 1-based to 0-based indexing.

In [ ]:
if not globals().get("CONFIRM_PUBLIC_WORKFLOW", False):
    raise RuntimeError("Execution disabled. Read docs/reproducibility.md and explicitly enable this workflow.")
examples = frame[frame.valid_label & frame.image_valid & frame.mask_shape_matches].groupby("class_name", sort=True).head(2)
fig, axes = plt.subplots(len(examples), 3, figsize=(10, 3 * len(examples)), squeeze=False)
for row_axes, (_, row) in zip(axes, examples.iterrows()):
    fields, _ = read_record(row)
    image, mask = fields["image"], fields["tumorMask"]
    row_axes[0].imshow(image, cmap="gray")
    row_axes[0].set_title(f"{row.class_name} | sample {row.sample_id}")
    row_axes[1].imshow(mask, cmap="gray", vmin=0, vmax=1)
    row_axes[1].set_title("Binary tumorMask")
    row_axes[2].imshow(image, cmap="gray")
    row_axes[2].imshow(np.ma.masked_where(mask == 0, mask), cmap="autumn", alpha=0.35, vmin=0, vmax=1)
    xy = fields["tumorBorder"].ravel().reshape(-1, 2) - 1
    xy = np.vstack([xy, xy[0]])
    row_axes[2].plot(xy[:, 0], xy[:, 1], color="cyan", linewidth=0.8)
    row_axes[2].set_title("Mask overlay + original border")
    for ax in row_axes:
        ax.axis("off")
fig.tight_layout()
fig.savefig(RESULTS / "dataset_examples.png", dpi=160)
plt.show()

## Validity checks and limitations
Phase 1 has no splits and cannot establish absence of train/test leakage. Missing/inconsistent PID and identical images across patients are initial risk indicators. Do not automatically merge/change patient IDs or apply cvind.mat.

In [ ]:
if not globals().get("CONFIRM_PUBLIC_WORKFLOW", False):
    raise RuntimeError("Execution disabled. Read docs/reproducibility.md and explicitly enable this workflow.")
checks = {
    "all_files_accounted_for": len(records) == len(frame) + len(auxiliary) + len(errors),
    "all_sample_ids_unique": summary["duplicate_sample_ids"] == 0,
    "all_labels_valid": summary["invalid_labels"] == 0,
    "all_pids_present": summary["missing_pids"] == 0,
    "consistent_patient_labels": not summary["inconsistent_pids"],
    "all_images_valid": summary["invalid_images"] == 0,
    "all_masks_present_nonempty_binary_aligned_shape": all(summary[k] == 0 for k in ["missing_masks", "empty_masks", "nonbinary_masks", "mask_shape_mismatches"]),
    "all_borders_valid": summary["invalid_borders"] == 0,
    "no_unreadable_files": not errors,
    "no_cross_patient_exact_duplicates": summary["cross_patient_duplicate_image_groups"] == 0,
}
display(pd.Series(checks))
(RESULTS / "validation_checks.json").write_text(json.dumps(checks, indent=2), encoding="utf-8")
assert checks["all_files_accounted_for"]
display(Markdown(f"**Observed result:** {len(frame)} readable images, {summary['unique_patients']} unique patient identifiers. Checks passed: {sum(checks.values())}/{len(checks)}. Any failed check requires review before splitting."))

## Historical conclusion and review boundary
Environment, structure, statistics, issues and examples were saved in results/. These are not model-performance results; no model was created/trained and Phase 2 had not begun. Compare observed counts with the reference and resolve patient-identity differences before approving the protocol. The historical next step was five patient-level folds with explicit PID intersection checks, after researcher review.
### Recorded findings
3,064 images, 233 patients; all ten validity checks passed. Glioma: 1,426 images/89 patients; Meningioma: 708/82; Pituitary: 930/62. There were 3,049 images at 512×512 and 15 at 256×256, int16, overall range 0–12,728. No missing/empty masks, unreadable files, inconsistent PID-label pairs or exact duplicate images. Slices per patient: mean 13.15, SD 7.66, median 13, range 1–38. Six examples were visually reviewed; alignment there is not a full anatomical audit. Patient-disjoint split validation belonged to Phase 2. These findings are historical, not newly executed.